#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "cost",
    "prd_line": "line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date",
    }

# Reading Data from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

df.display()

# Rename columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Trim string

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.display()

# Replace nulls with 0 value for cost column

In [0]:
df = df.withColumn("cost", F.when(F.col("cost").isNull(), 0).otherwise(F.col("cost")))

In [0]:
df.display()

# Write to Silver table

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("workspace.silver.crm_products")
)